# Práctica 3: Deploy de un Modelo de Machine Learning en la Nube

---

**Máster en Aplicaciones de Machine Learning**  
**Bloque 3 — Despliegue de Modelos**

---

## ¿Qué vas a hacer en esta práctica?

En esta práctica vas a recorrer el ciclo completo de despliegue de un modelo de Machine Learning:

1. **Entrenar un modelo** de clasificación sobre el dataset **Iris** usando el algoritmo que elijas
2. **Crear una API REST** con FastAPI que exponga el modelo
3. **Contenerizar** la aplicación con Docker
4. **Desplegar la API en la nube** usando Render o Railway
5. **Verificar** que la API funciona correctamente desde cualquier lugar

Al finalizar, entregarás la **URL pública** de tu API y este notebook completado. El profesor realizará peticiones directas a tu endpoint para evaluar la práctica.

---


## Datos de entrega

Completa esta sección antes de entregar:

**Nombre:** Jean Rodriguez

**URL de la API desplegada:** _[https://tu-api.onrender.com o https://tu-api.up.railway.app]_

**Plataforma usada:** Render

**Algoritmo utilizado:** Random Forest

**Repositorio GitHub:** https://github.com/jearod/practica-deploy-ml

---

## Estructura de archivos que debes crear

Crea una **nueva carpeta** en tu equipo (fuera de este repositorio) con la siguiente estructura. Estos archivos son los que subirás a GitHub y luego desplegarás:

```
practica-deploy-ml/
├── train_model.py       # Script para entrenar y guardar el modelo
├── app.py               # API de FastAPI
├── requirements.txt     # Dependencias del proyecto
├── Dockerfile           # Configuración del contenedor Docker
├── .dockerignore        # Archivos a excluir del contenedor (opcional)
├── model.pkl            # Modelo entrenado (se genera al ejecutar train_model.py)
└── README.md            # Descripción del proyecto (opcional)
```

> **Importante:** A lo largo del código que te encontrarás hay algunos #TODOS que dan pistas de lo que hay que hacer. No obstante, se pueden hacer más cosas que se tendrán en cuenta en la evaluación (ej: usar Pydantic para validar datos API, ...)

> **Importante II:** `model.pkl` debe estar en el repositorio porque Render/Railway lo necesita para ejecutar la API. Si tu modelo pesa mucho (+50MB), elige uno más sencillo

---

## PASO 1: Entrenar y guardar el modelo

### 1.1 El dataset: Iris

Todos usaréis el dataset **Iris**, incluido en scikit-learn. Contiene 150 muestras de flores con 4 medidas cada una, y el objetivo es clasificarlas en 3 especies:

| Feature | Descripción |
|---|---|
| `sepal_length` | Longitud del sépalo (cm) |
| `sepal_width` | Ancho del sépalo (cm) |
| `petal_length` | Longitud del pétalo (cm) |
| `petal_width` | Ancho del pétalo (cm) |

**Clases:** `0 = setosa`, `1 = versicolor`, `2 = virginica`

Lo que **sí puedes elegir** es el **algoritmo de clasificación**. Algunas opciones:

- `RandomForestClassifier`
- `LogisticRegression`
- `SVC`
- `GradientBoostingClassifier`
- `KNeighborsClassifier`

### 1.2 Escribe el código de entrenamiento

Completa el código. Lee los comentarios `# TODO` para saber qué debes implementar.

Ejecútalo aquí en el notebook para verificar que funciona antes de pasarlo al archivo `train_model.py`.

In [ ]:
# ============================================================
# PASO 1: Entrenar y guardar el modelo
# Archivo: train_model.py
# ============================================================

import joblib
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# Cargar el dataset Iris
iris = load_iris()
X, y = iris.data, iris.target

# -------------------------------------------------------
# TODO 1: Divide los datos en entrenamiento y prueba
# -------------------------------------------------------

# Tu código aquí:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.33, random_state=42)

# -------------------------------------------------------
# TODO 2: Elige y entrena un algoritmo de clasificación
# -------------------------------------------------------

# Tu código aquí:
model = LogisticRegression()
model.fit(X_train, y_train)

# -------------------------------------------------------
# TODO 3: Evalúa el modelo y muestra métricas
# -------------------------------------------------------

# Tu código aquí:


# -------------------------------------------------------
# TODO 4: Guarda el modelo como 'model.pkl'
# -------------------------------------------------------

# Tu código aquí:


print('¡Modelo guardado correctamente!')

ImportError: cannot import name 'LogisticRegression' from 'sklearn.model_selection' (/home/jearod/.local/lib/python3.10/site-packages/sklearn/model_selection/__init__.py)

### 1.3 Justifica tus decisiones

Responde brevemente a estas preguntas (2-3 frases cada una):

**¿Por qué elegiste ese algoritmo de clasificación? ¿Qué ventajas tiene sobre las otras opciones?**

_[Tu respuesta aquí]_

---

**¿Qué accuracy obtuviste? ¿Te parece un buen resultado para el dataset Iris?**

_[Tu respuesta aquí]_

---

**¿Probaste más de un algoritmo? Si es así, ¿cuál funcionó mejor y por qué crees que fue así?**

_[Tu respuesta aquí]_

---

## PASO 2: Crear la API con Fast API

La API debe tener **al menos estos tres endpoints**:

| Endpoint | Método | Descripción |
|---|---|---|
| `/` | GET | Información de la API: nombre, versión, algoritmo usado, ejemplo de llamada |
| `/predict` | POST | Recibe las 4 features de Iris en JSON y devuelve la especie predicha |
| `/health` | GET | Devuelve `{"status": "healthy"}` |

### 2.1 Formato del endpoint `/predict`

El endpoint `/predict` debe aceptar exactamente este formato JSON:

**Input:**
```json
{
    "sepal_length": 5.1,
    "sepal_width": 3.5,
    "petal_length": 1.4,
    "petal_width": 0.2
}
```

**Output:**
```json
{
    "prediction": "setosa",
    "prediction_index": 0,
    "probabilities": {
        "setosa": 0.97,
        "versicolor": 0.02,
        "virginica": 0.01
    },
    "confidence": 0.97,
    "status": "success"
}
```

> **Importante:** Respeta estos nombres de campo exactamente. El profesor usará este formato fijo para probar todas las APIs.

### 2.2 Implementa la API

Completa el código. Observa los comentarios `# TODO`.

In [ ]:
# ============================================================
# PASO 2: API de FastAPI
# Archivo: app.py
# ============================================================

import joblib
import numpy as np
import os


# Cargar modelo al arrancar la aplicación
model = joblib.load('model.pkl')

# Clases del dataset Iris (en el mismo orden que sklearn)
CLASSES = ['setosa', 'versicolor', 'virginica']


# -------------------------------------------------------
# TODO 5: Implementa el endpoint raíz '/'
# Debe devolver: nombre de la API, versión, algoritmo usado,
# las 4 features esperadas y un ejemplo de llamada a /predict
# -------------------------------------------------------
@app.route('/')
def home():
    # Tu código aquí:
    pass


# -------------------------------------------------------
# TODO 6: Implementa el endpoint '/predict'
# - Devuelve prediction, probabilities y status
# - Maneja errores: campos faltantes (KeyError) y otros (Exception)
# -------------------------------------------------------
@app.route('/predict', methods=['POST'])
def predict():
    # Tu código aquí:
    pass


# -------------------------------------------------------
# TODO 7: Implementa el endpoint '/health'
# Debe devolver {"status": "healthy"} con código 200
# -------------------------------------------------------
@app.route('/health')
def health():
    # Tu código aquí:
    pass


if __name__ == '__main__':
    # Render/Railway asigna el puerto mediante la variable PORT
    port = int(os.environ.get('PORT', 5000))
    app.run(host='0.0.0.0', port=port, debug=False)

### 2.4 Prueba la API en local

Antes de desplegar, verifica que la API funciona en tu máquina:

1. Abre una terminal y ejecuta: `python app.py`
2. Deja esa terminal abierta
3. Ejecuta la celda de abajo para hacer una petición a tu API local

> Si el servidor ya está corriendo en otro proceso, detén ese proceso primero.

In [ ]:
# ============================================================
# Prueba de la API en local
# Ejecuta primero: python app.py en una terminal
# ============================================================

import requests
import json

BASE_URL = 'http://localhost:XXXX'

# --- Prueba 1: endpoint raíz ---
print('=== GET / ===')


# --- Prueba 2: health check ---
print('=== GET /health ===')


# --- Prueba 3: predicción (Iris setosa) ---
print('=== POST /predict (setosa) ===')
test_data = {

}
response = requests.post(f'{BASE_URL}/predict', json=test_data)


**¿La API responde correctamente en local?**

_[Escribe aquí el output que obtuviste y confirma si es correcto]_

---

## PASO 3: Configurar Docker

### 3.1 Crea el Dockerfile

El Dockerfile le dice a Render/Railway cómo construir y ejecutar tu aplicación.

Crea un archivo llamado `Dockerfile` (sin extensión) en la raíz de tu proyecto con el contenido que defines aquí:

In [ ]:
# ============================================================
# Contenido del Dockerfile
# Copia este contenido a un archivo llamado 'Dockerfile'
# (sin extensión .txt ni .py)
# ============================================================

dockerfile_content = """
# TODO 11: Completa el Dockerfile
# Pistas:
#   - Usa python:3.11-slim como imagen base
#   - El directorio de trabajo debe ser /app
#   - El puerto lo asigna Render/Railway via variable PORT
#   - Usa gunicorn para servir la aplicación en producción
"""

print(dockerfile_content)

### 3.2 Crea el requirements.txt

Lista **todas** las dependencias de tu proyecto con sus versiones exactas.

> Tip: Ejecuta `pip freeze` en tu entorno virtual y selecciona solo las que realmente usas.

In [ ]:
# ============================================================
# Contenido del requirements.txt
# ============================================================

requirements_content = """
"""

print(requirements_content)

### 3.3 Prueba Docker en local (opcional pero recomendado)

Si tienes Docker instalado, prueba la imagen antes de desplegar. Ejecuta estos comandos en la terminal dentro de tu carpeta de proyecto:



**¿Funcionó Docker en local? ¿Encontraste algún error?**

_[Escribe aquí tu respuesta. Si no tienes Docker instalado, indica que saltaste este paso]_

---

## PASO 4: Subir a GitHub y Desplegar

### 4.1 Crear el repositorio en GitHub

1. Ve a [github.com](https://github.com) y crea un nuevo repositorio público
2. Nómbralo de forma descriptiva, por ejemplo: `ceste-practica-api`
3. Sube tu codigo


### 4.2 Desplegar en Render o Railway

Elige **una** de las dos plataformas:

---

#### Opción A: Render

1. Ve a [render.com](https://render.com) → Regístrate con tu cuenta de GitHub
2. **New** → **Web Service**
3. Conecta tu repositorio
4. Configuración:
   ```
   Name:        mi-api-ml (o el nombre que quieras)
   Environment: Docker
   Branch:      main
   Plan:        Free
   ```
5. Click en **Create Web Service**
6. Espera ~3-5 minutos a que termine el build

---

#### Opción B: Railway

1. Ve a [railway.app](https://railway.app) → Regístrate con GitHub
2. **New Project** → **Deploy from GitHub Repo**
3. Selecciona tu repositorio
4. Railway detecta el Dockerfile automáticamente
5. Ve a **Settings** → **Networking** → **Generate Domain**
6. Espera ~3-5 minutos al build

---

> **Nota:** La primera vez puede tardar más. Si el build falla, revisa los logs en la plataforma. Los errores más comunes son versiones de librerías incompatibles o el modelo no incluido en el repo.

---

## PASO 5: Verificar el despliegue

Una vez que el despliegue haya terminado, **cambia la URL** en la celda siguiente y ejecútala para verificar que todo funciona desde la nube.

In [ ]:
# ============================================================
# PASO 5: Verificar la API desplegada
# ============================================================

import requests
import json

# TODO 13: Cambia esta URL por la URL de TU API desplegada
BASE_URL = 'https://TU-API.onrender.com'  # <-- CAMBIA ESTO

print(f'Probando API en: {BASE_URL}')
print('=' * 50)

# --- Verificar que la API responde ---
try:
    response = requests.get(f'{BASE_URL}/health', timeout=60)
    print(f'Health check: {response.status_code}')
    print(json.dumps(response.json(), indent=2))
except requests.exceptions.Timeout:
    print('⚠️  Timeout: El servicio puede estar en cold start. Espera 30 segundos y vuelve a intentarlo.')
except Exception as e:
    print(f'❌ Error: {e}')

print()

# --- Verificar el endpoint raíz ---
try:
    response = requests.get(f'{BASE_URL}/', timeout=60)
    print(f'Endpoint raíz: {response.status_code}')
    print(json.dumps(response.json(), indent=2))
except Exception as e:
    print(f'❌ Error: {e}')

In [ ]:
# ============================================================
# Prueba de predicción en la API desplegada
# Los 3 ejemplos cubren las 3 especies del dataset Iris
# ============================================================

test_cases = [
    {
        'description': 'Iris setosa',
        'data': {'sepal_length': 5.1, 'sepal_width': 3.5, 'petal_length': 1.4, 'petal_width': 0.2}
    },
    {
        'description': 'Iris versicolor',
        'data': {'sepal_length': 7.0, 'sepal_width': 3.2, 'petal_length': 4.7, 'petal_width': 1.4}
    },
    {
        'description': 'Iris virginica',
        'data': {'sepal_length': 6.3, 'sepal_width': 3.3, 'petal_length': 6.0, 'petal_width': 2.5}
    },
]

print(f'Probando predicciones en: {BASE_URL}/predict')
print('=' * 50)

for test in test_cases:
    print(f"\n--- {test['description']} ---")
    try:
        response = requests.post(
            f'{BASE_URL}/predict',
            json=test['data'],
            timeout=60
        )
        print(f'Status: {response.status_code}')
        result = response.json()
        print(json.dumps(result, indent=2))
        # Verificar que la predicción es correcta
        expected = test['description'].split()[-1].lower()  # 'setosa', 'versicolor' o 'virginica'
        predicted = result.get('prediction', '').lower()
        ok = '✅' if predicted == expected else '❌'
        print(f'{ok} Esperado: {expected} | Predicho: {predicted}')
    except Exception as e:
        print(f'❌ Error: {e}')

### 5.1 Resultado de las pruebas

**¿Todas las pruebas pasaron correctamente?**

_[Sí / No. Si no, describe qué falló y cómo lo resolviste]_

---

**Captura de pantalla del dashboard de Render/Railway con el estado "Live":**

_[Inserta aquí una imagen del dashboard mostrando que el servicio está activo]_

> Para insertar una imagen en Jupyter: menú Edit → Insert Image, o usa la sintaxis `![descripcion](ruta_imagen.png)`

---

## PASO 6: Reflexión final

Responde las siguientes preguntas con tus propias palabras (3-5 frases por pregunta):

**1. ¿Qué diferencia hay entre ejecutar un modelo en local y tenerlo desplegado en la nube?**

_[Tu respuesta aquí]_

---

**2. ¿Para qué sirve Docker en este contexto? ¿Qué problema resuelve?**

_[Tu respuesta aquí]_

---

**3. ¿Qué limitaciones tiene el plan gratuito de Render/Railway para un modelo en producción real?**

_[Tu respuesta aquí]_

---

**4. ¿Qué mejorarías en tu API si la pusieras en producción real?**

_[Tu respuesta aquí]_

---

## Checklist de entrega

Antes de entregar, verifica que has completado todo:

- [ ] Rellené mis datos de entrega al principio del notebook
- [ ] El modelo Iris se entrena correctamente (Paso 1)
- [ ] Justifiqué la elección del algoritmo
- [ ] La API tiene los tres endpoints: `/`, `/predict`, `/health`
- [ ] La API acepta el formato JSON fijo con las 4 features de Iris
- [ ] La API funciona en local (Paso 2.4)
- [ ] El Dockerfile está completo y correcto
- [ ] El `requirements.txt` incluye todas las dependencias
- [ ] El código está en un repositorio público de GitHub
- [ ] La API está desplegada y accesible públicamente
- [ ] Los 3 test cases de predicción pasan correctamente (Paso 5)
- [ ] Incluí una captura de pantalla del dashboard
- [ ] Respondí las preguntas de reflexión

---

## Entrega

Entrega a través del campus virtual:

1. **Este notebook** completado (`.ipynb`)
2. **La URL pública** de tu API (el profesor hará llamadas directas para evaluarla)
3. **La URL del repositorio** de GitHub

> El profesor probará tu API con los mismos 3 ejemplos de este notebook (setosa, versicolor, virginica). Asegúrate de que las 3 predicciones son correctas antes de entregar.